# Train XGBoost Risk Prediction Model

Trains an XGBoost regressor to predict infrastructure risk scores
based on weather, location, and historical factors.

## Imports

In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/nihesh/Nihesh/data-dynamo/ml/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <58FE87DD-A5B4-3D80-BC4B-11FC831B9707> /Users/nihesh/Nihesh/data-dynamo/ml/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt

## Configuration

In [ ]:
# Feature columns (excluding target)
FEATURE_COLUMNS = [
    "rainfall_mm",
    "temperature_c",
    "humidity_pct",
    "is_monsoon",
    "latitude",
    "longitude",
    "road_type_highway",
    "road_type_urban",
    "road_type_rural",
    "traffic_density",
    "issue_count_30d",
    "is_hotspot",
    "resolution_rate",
    "days_since_last_issue",
    "population_density",
]

TARGET_COLUMN = "risk_score"

print(f"Number of features: {len(FEATURE_COLUMNS)}")
print(f"Target: {TARGET_COLUMN}")

## Load Training Data

In [ ]:
def load_training_data(data_path: Path) -> pd.DataFrame:
    """
    Load training data from JSON file.

    Args:
        data_path: Path to training data JSON

    Returns:
        DataFrame with features and target
    """
    with open(data_path, "r") as f:
        data = json.load(f)

    return pd.DataFrame(data)

models_dir = Path("../models")
data_path = models_dir / "risk_training_data.json"

# Check if training data exists
if not data_path.exists():
    print("Training data not found. Please run generate_risk_data.ipynb first!")
else:
    print("Loading training data...")
    df = load_training_data(data_path)
    print(f"Total samples: {len(df)}")
    print(f"\nDataFrame shape: {df.shape}")
    df.head()

## Explore Data

In [ ]:
print("Dataset Info:")
print(df.info())

print("\nStatistical Summary:")
df.describe()

In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

all_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
for i, col in enumerate(all_columns):
    ax = axes[i]
    ax.hist(df[col], bins=30, edgecolor='black', alpha=0.7)
    ax.set_title(col)
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

# Hide empty subplot
axes[-1].axis('off')

plt.tight_layout()
plt.show()

## Prepare Data for Training

In [ ]:
# Prepare features and target
X = df[FEATURE_COLUMNS].values
y = df[TARGET_COLUMN].values

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully")
print(f"Training data mean (should be ~0): {X_train_scaled.mean():.6f}")
print(f"Training data std (should be ~1): {X_train_scaled.std():.6f}")

## Train XGBoost Model

In [ ]:
print("Training XGBoost model...")

model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=True
)

print("\nTraining complete!")

## Evaluate Model

In [ ]:
print("Evaluating model...")

y_pred = model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n" + "="*50)
print("TEST METRICS")
print("="*50)
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")
print("="*50)

In [ ]:
# Visualize predictions vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.5, s=10)
axes[0].plot([0, 1], [0, 1], 'r--', label='Perfect Prediction')
axes[0].set_xlabel('Actual Risk Score')
axes[0].set_ylabel('Predicted Risk Score')
axes[0].set_title(f'Predicted vs Actual (R² = {r2:.4f})')
axes[0].legend()

# Residuals histogram
residuals = y_test - y_pred
axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Residuals Distribution (MAE = {mae:.4f})')

plt.tight_layout()
plt.show()

## Feature Importance

In [ ]:
print("Feature Importance:")
importance = dict(zip(FEATURE_COLUMNS, model.feature_importances_))
sorted_importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)

for feature, imp in sorted_importance:
    print(f"  {feature}: {imp:.4f}")

In [ ]:
# Visualize feature importance
features = [x[0] for x in sorted_importance]
importances = [x[1] for x in sorted_importance]

plt.figure(figsize=(10, 8))
plt.barh(features[::-1], importances[::-1], color='steelblue', edgecolor='black')
plt.xlabel('Feature Importance')
plt.title('XGBoost Feature Importance for Risk Prediction')
plt.tight_layout()
plt.show()

## Save Model and Artifacts

In [ ]:
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

# Save model and scaler
model_path = models_dir / "risk_model.joblib"
scaler_path = models_dir / "risk_scaler.joblib"

joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)

print(f"Model saved to: {model_path}")
print(f"Scaler saved to: {scaler_path}")

In [ ]:
# Save metrics
metrics = {
    "rmse": float(rmse),
    "mae": float(mae),
    "r2": float(r2),
    "training_samples": len(X_train),
    "test_samples": len(X_test),
    "feature_importance": {k: float(v) for k, v in importance.items()},
    "model_params": model.get_params(),
}

metrics_path = models_dir / "risk_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics saved to: {metrics_path}")

In [ ]:
# Save feature columns for inference
config = {
    "feature_columns": FEATURE_COLUMNS,
    "target_column": TARGET_COLUMN,
}
config_path = models_dir / "risk_model_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Config saved to: {config_path}")

## Test Inference

In [ ]:
# Test prediction with sample data
sample_input = {
    "rainfall_mm": 150.0,
    "temperature_c": 28.0,
    "humidity_pct": 85.0,
    "is_monsoon": 1,
    "latitude": 19.0,
    "longitude": 72.8,
    "road_type_highway": 1,
    "road_type_urban": 0,
    "road_type_rural": 0,
    "traffic_density": 0.8,
    "issue_count_30d": 25,
    "is_hotspot": 1,
    "resolution_rate": 0.4,
    "days_since_last_issue": 5,
    "population_density": 0.9,
}

# Convert to array
input_array = np.array([[sample_input[col] for col in FEATURE_COLUMNS]])
input_scaled = scaler.transform(input_array)
prediction = model.predict(input_scaled)[0]

print("Sample Prediction Test")
print("="*50)
print("Input features (high risk scenario):")
for k, v in sample_input.items():
    print(f"  {k}: {v}")
print(f"\nPredicted Risk Score: {prediction:.4f}")
print(f"Risk Level: {'CRITICAL' if prediction >= 0.8 else 'HIGH' if prediction >= 0.6 else 'MEDIUM' if prediction >= 0.3 else 'LOW'}")